# Set Up Notebook

In [ ]:
##########
# IMPORT #
##########

# Data processing and math
import pandas as pd
import numpy as np

# Statistics
import scipy.stats as stats
import statsmodels.api as sm
from statsmodels.formula.api import ols, mixedlm
from statsmodels.stats.anova import AnovaRM
from statsmodels.tools.sm_exceptions import ConvergenceWarning
from statsmodels.stats.multitest import multipletests

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Display handling
import warnings
from IPython.display import display, HTML


#################
# CONFIGURATION #
#################

# Suppress warnings
warnings.filterwarnings('ignore', category = UserWarning)
warnings.filterwarnings('ignore', category = RuntimeWarning)
warnings.filterwarnings('ignore', category = ConvergenceWarning)

# Configure display
display(HTML("<style>.output_scroll { height: auto !important; max-height: none !important; }</style>"))

# Set global plotting style
sns.set_theme(style = "whitegrid")
%matplotlib inline


##########
# LABELS #
##########
labels_measure = {
    'measure_action':  'Responsibility for Action',
    'measure_beliefs': 'Responsibility for Adult Moral Beliefs',
    'measure_self':    'True Self'
}

labels_gender = {
    '1': 'Female',
    '2': 'Male',
    '3': 'Non-binary',
    '4': 'Prefer not to say'
}

labels_education = {
    '1': 'Less than high school',
    '2': 'High school diploma or equivalent',
    '3': 'Associate degree (e.g., AA or AS)',
    '4': 'Bachelor’s degree (e.g., BA or BSC)',
    '5': 'Master’s degree (e.g., MA or MSc)',
    '6': 'Professional degree (e.g., JD or MD)',
    '7': 'Doctorate (e.g., PhD or EdD)'
}


#################
# VISUALIZATION #
#################
palette_main = {
    "Good":      "#0072B2", "Bad":    "#D55E00",
    "Activist":  "#56B4E9", "Bigot":  "#E69F00",
    "Help":      "#F0E442", "Harm":   "#882255",
    "Young":     "#66c2a5", "Middle": "#fc8d62", "Old": "#8da0cb"
}

# Transform Data

In [ ]:
# Load data
df = pd.read_csv('blame_praise_self_study_2_data.csv')

# Define factors
def get_factors(n):
    mapping = {
         1: ("Bad", "Reflected", "Activist", "Help", "Young"),
         2: ("Bad", "Reflected", "Activist", "Help", "Middle"),
         3: ("Bad", "Reflected", "Activist", "Help", "Old"),
         4: ("Bad", "Reflected", "Activist", "Harm", "Young"),
         5: ("Bad", "Reflected", "Activist", "Harm", "Middle"),
         6: ("Bad", "Reflected", "Activist", "Harm", "Old"),
         7: ("Bad", "Reflected", "Bigot", "Help", "Young"),
         8: ("Bad", "Reflected", "Bigot", "Help", "Middle"),
         9: ("Bad", "Reflected", "Bigot", "Help", "Old"),
        10: ("Bad", "Reflected", "Bigot", "Harm", "Young"),
        11: ("Bad", "Reflected", "Bigot", "Harm", "Middle"),
        12: ("Bad", "Reflected", "Bigot", "Harm", "Old"),
        13: ("Good", "Reflected", "Activist", "Help", "Young"),
        14: ("Good", "Reflected", "Activist", "Help", "Middle"),
        15: ("Good", "Reflected", "Activist", "Help", "Old"),
        16: ("Good", "Reflected", "Activist", "Harm", "Young"),
        17: ("Good", "Reflected", "Activist", "Harm", "Middle"),
        18: ("Good", "Reflected", "Activist", "Harm", "Old"),
        19: ("Good", "Reflected", "Bigot", "Help", "Young"),
        20: ("Good", "Reflected", "Bigot", "Help", "Middle"),
        21: ("Good", "Reflected", "Bigot", "Help", "Old"),
        22: ("Good", "Reflected", "Bigot", "Harm", "Young"),
        23: ("Good", "Reflected", "Bigot", "Harm", "Middle"),
        24: ("Good", "Reflected", "Bigot", "Harm", "Old")
    }
    return mapping.get(n)

# Reshape wide to long
contexts = ['homophobia', 'racism', 'sexism']
measure_mapping = {'pAction': 'measure_action', 'pBeliefs': 'measure_beliefs', 'agreement': 'measure_self'}
long_data = []

for _, row in df.iterrows():
    p_id = row['ID']
    for ctx in contexts:
        for i in range(1, 25):
            col_base = f'{ctx}.{i}'
            if f'{col_base}-pAction' in df.columns and pd.notna(row[f'{col_base}-pAction']):
                up_f, refl_f, bel_f, val_f, age_f = get_factors(i)
                trial_data = {
                    'ID': p_id, 'Context': ctx, 'Upbringing': up_f,
                    'Reflection': refl_f, 'Belief': bel_f, 'Action': val_f, 'Agent_Age': age_f
                }
                for old_suffix, new_name in measure_mapping.items():
                    val = pd.to_numeric(row[f'{col_base}-{old_suffix}'], errors = 'coerce')
                    
                    # Center scale
                    trial_data[new_name] = val - 7 if pd.notna(val) else np.nan
                long_data.append(trial_data)

df_long = pd.DataFrame(long_data).dropna(subset = ['measure_action'])

# Create DataFrames
demo_cols = ['ID', 'Gender', 'Age', 'Education', 'politics', 'determinism', 'freeWill', 'MoralRealism', 'Duration (in seconds)']
existing_demo_cols = [c for c in demo_cols if c in df.columns]
df_robust = df_long.merge(df[existing_demo_cols], on = 'ID', how = 'left')

# Rename variables
df_robust = df_robust.rename(columns = {
    'politics':              'Political_Orientation',
    'determinism':           'Social_Determinism',
    'freeWill':              'Free_Will',
    'Duration (in seconds)': 'Duration_Seconds',
    'MoralRealism':          'Moral_Realism'
})

# Log-transform duration to handle distribution skewness
if 'Duration_Seconds' in df_robust.columns:
    df_robust['Duration_Seconds'] = pd.to_numeric(df_robust['Duration_Seconds'], errors = 'coerce')
    df_robust['Log_Duration'] = np.log1p(df_robust['Duration_Seconds'])

# Apply z-standardization of continuous covariates
covs = ['Age', 'Political_Orientation', 'Social_Determinism', 'Free_Will', 'Log_Duration']
existing_covs = [c for c in covs if c in df_robust.columns]

# Ensure numeric types before scaling
df_robust[existing_covs] = df_robust[existing_covs].apply(pd.to_numeric, errors = 'coerce')

# Apply z-score formula
for c in existing_covs:
    
    # Set standardized name
    target_col = 'Duration_z' if c == 'Log_Duration' else f'{c}_z'
    df_robust[target_col] = (df_robust[c] - df_robust[c].mean()) / df_robust[c].std()

print(f"Transformation complete ({len(df_robust)} Observations).")

# Sociodemographics

In [ ]:
df_robust['Gender_Labeled'] = df_robust['Gender'].astype(str).map(labels_gender)
df_robust['Education_Labeled'] = df_robust['Education'].astype(str).map(labels_education)
df_robust['Age'] = pd.to_numeric(df_robust['Age'], errors = 'coerce')

for col in ['Gender_Labeled', 'Education_Labeled']:
    print(f"\n{col}:")
    display(df_robust[col].value_counts().to_frame('Frequency'))

display(df_robust['Age'].describe().to_frame('Value').round(3))

# Background Variables

In [ ]:
# Define background variables
background_vars = ['Political_Orientation', 'Social_Determinism', 'Free_Will', 'Moral_Realism']

# Get summary
background_summary = df_robust[background_vars].describe().T[['mean', 'std', 'min', 'max']].round(3)
display(background_summary)

# Map titles to labels
background_titles = {
    'Political_Orientation': 'Political Orientation\n(1 = Left, 7 = Right)',
    'Social_Determinism':    'Social Determinism\n(1 = Low, 7 = High)',
    'Free_Will':             'Free Will\n(1 = Low, 7 = High)',
    'Moral_Realism':         'Moral Realism\n(1 = Objectivist, 2 = Relativist, 3 = Unsure)'
}

# Generate graphs
plt.figure(figsize = (22, 5))

for i, col in enumerate(background_vars, 1):
    plt.subplot(1, 4, i)
    sns.countplot(data = df_robust, x = col, color = 'salmon', edgecolor = 'black')
    
    # Set style
    plt.title(background_titles[col], fontweight = 'bold', pad = 15)
    plt.xlabel('Value')
    plt.ylabel('Frequency')
    plt.ylim(0, df_robust[col].value_counts().max() * 1.1)

# Export graphs
filename = 'blame_praise_self_study_2_background.png'
plt.tight_layout()
plt.savefig(filename, dpi = 300)
plt.show()
print(f"Graph saved as '{filename}'.")

# Calculate Descriptive Statistics

In [ ]:
# Define group factors
group_factors = ['Upbringing', 'Belief', 'Action', 'Agent_Age']

# Define dependent variables
dependent_vars = ['measure_action', 'measure_beliefs', 'measure_self']

# Calculate descriptive statistics
descriptive_stats = df_long.groupby(group_factors)[dependent_vars].agg(['mean', 'std', 'count']).round(3)

# Display results
display(descriptive_stats)

# Perform ANOVAs

In [ ]:
for dv in dependent_vars:
    
    # Print headers
    print(f"\nANOVA ({labels_measure.get(dv, dv)})")
    
    # Define formula
    formula = f"{dv} ~ C(Upbringing) * C(Belief) * C(Action) * C(Agent_Age)"
    
    # Fit model
    model = ols(formula, data = df_long).fit()
    
    # Run ANOVAs
    aov = sm.stats.anova_lm(model, typ = 2)
    
    # Calculate effect sizes
    aov['partial_eta_sq'] = aov['sum_sq'] / (aov['sum_sq'] + aov.loc['Residual', 'sum_sq'])
    
    # Rename columns
    aov.index.name = 'Source'
    aov = aov.rename(columns = {'sum_sq': 'SS', 'df': 'DF', 'PR(>F)': 'p_unc'})
    
    # Multiple testing correction
    mask = aov['p_unc'].notna()
    _, p_corr, _, _ = multipletests(aov.loc[mask, 'p_unc'], alpha = 0.05, method = 'fdr_bh')
    aov.loc[mask, 'p_corr'] = p_corr
    
    # Display results
    display(aov[['SS', 'DF', 'F', 'p_unc', 'p_corr', 'partial_eta_sq']].round(3))

# Perform ANCOVAs

In [ ]:
# Define covariates
covariates = [
    "C(Gender)",
    "Age",
    "C(Education)",
    "Political_Orientation",
    "Social_Determinism",
    "Free_Will",
    "Duration_z",
    "C(Moral_Realism)"
]

for dv in dependent_vars:
    
    # Print header
    print(f"\nANCOVA ({labels_measure.get(dv, dv)})")
    
    # Define formula
    formula = f"{dv} ~ C(Upbringing) * C(Belief) * C(Action) * C(Agent_Age) + " + " + ".join(covariates)

    # Fit model
    model = ols(formula, data = df_robust).fit()
    
    # Run ANCOVAs
    aov_table = sm.stats.anova_lm(model, typ = 2)
    
    # Calculate effect sizes
    res_ss = aov_table.loc['Residual', 'sum_sq']
    aov_table['partial_eta_sq'] = aov_table['sum_sq'] / (aov_table['sum_sq'] + res_ss)
    
    # Rename columns
    aov_table = aov_table.rename(columns = {'sum_sq': 'SS', 'df': 'DF', 'PR(>F)': 'p_unc'})
    
    # Multiple testing correction
    mask = aov_table['p_unc'].notna()
    _, p_corr, _, _ = multipletests(aov_table.loc[mask, 'p_unc'], alpha = 0.05, method = 'fdr_bh')
    aov_table.loc[mask, 'p_corr'] = p_corr
    
    # Display results
    display(aov_table[['SS', 'DF', 'F', 'p_unc', 'p_corr', 'partial_eta_sq']].round(3))

# Generate Histogram

In [ ]:
# Create graph
plt.figure(figsize = (10, 6))

sns.histplot(df_robust['Duration_Seconds'], bins = 50, kde = True, color = '#0072B2')

# Add vertical line for median
median_dur = df_robust['Duration_Seconds'].median()
plt.axvline(median_dur,
            color = '#D55E00',
            linestyle = '--',
            lw = 2,
            label = f'Median: {median_dur:.0f} sec')

# Set labels and title
plt.title("Response Durations", fontsize = 14)
plt.xlabel("Seconds", fontsize = 12)
plt.ylabel("Frequency", fontsize = 12)
plt.legend()

# Limit x-axis to 95th percentile
plt.xlim(0, df_robust['Duration_Seconds'].quantile(0.95))

# Export graph
filename = 'blame_praise_self_study_2_duration.png'
plt.savefig(filename, dpi = 300, bbox_inches = 'tight')
plt.show()
print(f"Graph saved as '{filename}'.")

# Generate Bar Plots

In [ ]:
# Define combinations
combinations = [
    ("Agent_Age",  "Upbringing", "Action"),
    ("Agent_Age",  "Belief",     "Action"),
    ("Upbringing", "Belief",     "Action")
]

# Create graphs
for measure in dependent_vars:
    current_label = labels_measure.get(measure, measure)
    
    for x_var, hue_var, col_var in combinations:
        g = sns.catplot(data = df_long,
                        x = x_var,
                        y = measure,
                        hue = hue_var,
                        col = col_var,
                        kind = "bar",
                        capsize = .05,
                        errorbar = "ci",
                        palette = palette_main,
                        height = 5
                       )
        
        # Display Agent_Age as Age
        display_x = "Age" if x_var == "Agent_Age" else x_var
        display_hue = "Age" if hue_var == "Agent_Age" else hue_var
        
        # Set style
        g.set_axis_labels(display_x, "Mean Score (–6 to +6)")
        g.set_titles("{col_name}")
        
        g.fig.suptitle(f"{display_x} × {display_hue} ({current_label})", y = 1.05, fontsize = 14)
        
        for ax in g.axes.flat:
            ax.axhline(0, color = 'black', linewidth = 1)
            ax.set_ylim(-6, 6)
        
        # Export graphs
        filename = f'blame_praise_self_study_2_bar_plot_{measure}_{x_var}_{hue_var}.png'.lower()
        g.savefig(filename, dpi = 300, bbox_inches = 'tight')
        plt.show()
        print(f"Graph saved as '{filename}'.")

# Generate Point Plots

In [ ]:
# Define interactions
interactions = [
    ("Agent_Age",  "Upbringing", "Action"),
    ("Agent_Age",  "Belief",     "Action"),
    ("Upbringing", "Belief",     "Action")
]

# Iterate through dependent variables and interactions
for measure in dependent_vars:
    current_label = labels_measure.get(measure, measure)
    
    for x_var, hue_var, col_var in interactions:
        
        # Create graphs
        g = sns.catplot(data = df_long,
                        x = x_var,
                        y = measure,
                        hue = hue_var,
                        col = col_var,
                        kind = "point",
                        capsize = .15,
                        palette = palette_main,
                        height = 5,
                        aspect = 1
                       )
        
        # Display Agent_Age as Age
        display_x = "Age" if x_var == "Agent_Age" else x_var
        display_hue = "Age" if hue_var == "Agent_Age" else hue_var
        
        # Set axis labels and titles
        g.set_axis_labels(display_x, f"{current_label}")
        g.set_titles("{col_name}")
        
        # Add main titles
        g.fig.suptitle(f"{display_x} × {display_hue} ({current_label})", y = 1.05, fontsize = 14)
        
        # Format axes
        for ax in g.axes.flat:
            
            # Add horizontal zero lines
            ax.axhline(0, color = 'black', linewidth = 1, linestyle = '--', alpha = 0.5)
            
            # Set scale limits
            ax.set_ylim(-6, 6)
        
        # Export graphs
        filename = f'blame_praise_self_study_2_interaction_plot_{measure}_{x_var}_{hue_var}.png'.lower()
        g.savefig(filename, dpi = 300, bbox_inches = 'tight')
        plt.show()
        print(f"Graph saved as '{filename}'.")

# Generate Violin Plots

In [ ]:
# Reshape DataFrame from wide to long for categorical plotting
df_melted = df_long.melt(id_vars    = ['Action', 'Upbringing', 'Belief', 'Agent_Age'],
                         value_vars = dependent_vars,
                         var_name   = 'Measure',
                         value_name = 'Score')

# Rename column
df_melted = df_melted.rename(columns={'Agent_Age': 'Age'})

# Map measures to labels
df_melted['Measure'] = df_melted['Measure'].replace({
    'measure_action':  'Responsibility for Action',
    'measure_beliefs': 'Responsibility for Adult Moral Beliefs',
    'measure_self':    'True Self'
})

# Map factors to color palettes
dimensions = [
    ('Upbringing', palette_main),
    ('Action',     palette_main),
    ('Belief',     palette_main),
    ('Age',        palette_main)
]

# Create graphs
for factor_name, palette in dimensions:
    plt.figure(figsize = (10, 6))
    
    # Don't split violins for age
    can_split = (factor_name != 'Age')
    
    # Split violins
    sns.violinplot(data = df_melted,
                   x = "Measure",
                   y = "Score",
                   hue = factor_name,
                   split = can_split,
                   inner = "quart",
                   palette = palette)
    
    # Set style
    plt.axhline(0, color = 'black', linewidth = 1, linestyle = '--', alpha = 0.5)
    plt.title(f"{factor_name}", fontsize = 14)
    plt.ylim(-7, 7)
    plt.ylabel("Score (–6 to +6)")
    plt.xlabel("")
    
    # Export graphs
    filename_factor = 'age' if factor_name == 'Age' else factor_name.lower()
    filename = f'blame_praise_self_study_2_violin_plot_{filename_factor}.png'
    
    plt.savefig(filename, dpi = 300, bbox_inches = 'tight')
    plt.show()
    print(f"Graph saved as '{filename}'.")